# 頂点被覆問題 (Vertex Cover)

グラフ理論の数学的分野において、グラフの頂点被覆(ノード被覆と呼ばれることもあります)とは、グラフの各辺が集合内の少なくとも1つの頂点に接続しているような頂点の集合のことです。

https://en.wikipedia.org/wiki/Vertex_cover

![vertex cover](https://github.com/mdrft/Wildqat/blob/master/examples_ja/img/016_1.png?raw=1)

## インストール

まだblueqatをインストールしていない場合は、インストールしてください。

```bash
pip install git+https://github.com/blueqat/blueqatSDK
```

それでは始めましょう。

In [1]:
import numpy as np
from blueqat.utils import Vqe, QaoaAnsatz
from blueqat.utils import qubo_bit as q

## QUBO
今回のコスト関数は

$ H = H_{A} + H_{B} $

であり、$H_{A} と H_{B}$ は

$ \displaystyle H _ { A } = A \sum _ { u v \in E } \left( 1 - x _ { u } \right) \left( 1 - x _ { v } \right)$

$ \displaystyle H _ { B } = B \sum _ { v } x _ { v }$

であり、各 $x_{u}, x_{v}$ は、その頂点が選ばれていれば1、そうでなければ0を表します。

すると

$ \displaystyle H _ { A } = A \sum _ { u v \in E } \left( 1 - x _ { u } - x _ { v } + x_{u}x_{v}\right)$

定数項は無視でき、バイナリ変数の規則から次のようになります。

$ \displaystyle H_{A} = A \sum _ { u v \in E } \left( - x_{u}x_{u} - x_{v}x_{v} + x_{u}x_{v}\right) $

また、2つ目のコスト関数として

$ \displaystyle H_{B} = B \sum _ { u,v: u = v } x_{u}x_{v}$

があります。

## コーディングと計算
無向グラフを次のように定義します。

In [2]:
edge_def = [
    [1,5],      # vertex connected to (0) are (1) and (5)
    [2,5],      # vertex connected to (1) are (2) and (5)
    [3,5],      #  :
    [4],        #  :
    [5,6,7,8],
    [6,7],
    [7],
    [],
    []
]

In [3]:
A = 1.0
B = 0.9
def get_qubo(edges):
    Q = 0

    for u in range(len(edges)):
        for v in range(u, len(edges)):
            if u == v:
                Q += B*q(u)*q(v)
            if v in edges[u]:    #if xu and xv are connected each other or not
                Q +=  +A*q(u)*q(v)
                Q +=  -A*q(u)*q(u)
                Q +=  -A*q(v)*q(v)

    return Q

結果を確認するための関数を作りましょう。

5回試してみましょう。

In [4]:
h = get_qubo(edge_def)
# Note: in the new SDK, `step` is the number of variational QAOA layers
# actually optimized by gradient descent (unlike the old Trotter-schedule
# semantics), so a small step count is used here to keep runtime reasonable.
step = 5
result = Vqe(QaoaAnsatz(h, step)).run(max_iter=100)
print(result.most_common(12))

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


(((0, 1, 1, 0, 0, 1, 1, 1, 0), 0.03481306340954693), ((0, 1, 1, 0, 0, 1, 1, 1, 1), 0.029998864269884087), ((0, 1, 1, 0, 1, 1, 1, 0, 1), 0.026637312567605726), ((0, 1, 1, 0, 1, 1, 0, 1, 1), 0.02663731256760571), ((0, 0, 1, 0, 0, 1, 1, 1, 0), 0.025416614700927608), ((0, 1, 1, 0, 0, 1, 1, 0, 1), 0.02448908044597583), ((0, 1, 1, 0, 0, 1, 0, 1, 1), 0.0244890804459758), ((0, 1, 0, 1, 0, 1, 1, 1, 0), 0.022469022338553878), ((0, 0, 1, 0, 0, 1, 1, 1, 1), 0.02116773348153072), ((0, 1, 0, 0, 1, 1, 1, 0, 1), 0.021017981661863), ((0, 1, 0, 0, 1, 1, 0, 1, 1), 0.021017981661862976), ((0, 1, 1, 0, 1, 1, 0, 0, 1), 0.020948241401793136))
